# 99_explore — optional read-only exploration support

Use this optional support notebook for discovery, profiling, troubleshooting, investigation, and ad hoc analysis. The required delivery path remains: `01_governance` → `02_pipeline` → `01_governance`.

`99_explore` can read selected agreement context and existing catalogue context to help you investigate a source table. It does **not** approve agreements, enforce guardrails, write pipeline metadata, register delivery state, promote outputs, or mutate governance metadata.

Keep repeatable transformation logic in `02_pipeline`. Keep approval and review workflows in `01_governance`.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
| v0.1.0 |  Voyce| 13 Jul 2026 | 


# FabricOps v0.1.0 onwards

## 01 Configure environment


In [ ]:
%run 00_env_config


## 02 Import functions


In [ ]:
from fabricops_kit import (
    # FabricOps v0.1.0 onwards
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)

In [ ]:
from fabricops_kit import (
    # FabricOps v0.2.0 onwards
    profile_dataframe,
    widget_view_data_catalogue,
)

## Example Read from files

For Lakehouse file helpers, paths are relative to the configured Lakehouse **Files** area. For example, `input/orders.csv` resolves under `Files/input/orders.csv`; do not add a leading `Files/` prefix in these helper calls.


In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - CSV uses Spark CSV reader options.

# Example CSV file from Source lakehouse:
source_df = read_lakehouse_csv(
     "orders.csv",          # This is relative path after Files/ so if you have folders do it 
     target="source",       # This is the name of your lakehouse you defined in 00_env_config
     spark_session=spark,
     header=True,
     inferSchema=True,
 )
display(source_df)

In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - Excel uses pandas.read_excel options, then converts to Spark DataFrame.

# Example EXCEL file from Source lakehouse
source_df = read_lakehouse_excel(
     "products.xlsx",
     target="source",
     sheet_name="products",
     spark_session=spark,
 )
display(source_df)


In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - Parquet uses Spark Parquet reader options.

# Example PARQUET file from Source lakehouse
source_df = read_lakehouse_parquet(
     "customers.parquet",
     target="source",
     spark_session=spark,
 )
display(source_df)


### Example write/read Lakehouse


In [ ]:
# Optional example — write current source_df to unified lakehouse.
# Assumes source_df already exists from CSV / Excel / Parquet / Lakehouse read above.

write_lakehouse_table(
    source_df,
    table_name="smoke_test_source_df",
    target="unified",
    mode="overwrite",
)

# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - read table uses Spark Delta reader options.

lakehouse_df = read_lakehouse_table(
    table_name="smoke_test_source_df",
    target="unified",
    spark_session=spark,
)

display(lakehouse_df)

### Example write/read Warehouse


In [ ]:
# Optional example — write current source_df to Product warehouse.
# Assumes source_df already exists from CSV / Excel / Parquet / Lakehouse read above.

write_warehouse_table(
    source_df,
    schema="dbo",
    table_name="smoke_test_source_df",
    target="product",
    mode="overwrite",
)

# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
#  - Warehouse table uses the Fabric Warehouse Spark connector to read a full schema.table. 
#  - This reads and returns us the full table under the hood so like select *, so if we read big table it will take a long time

warehouse_df = read_warehouse_table(
    schema="dbo",
    table_name="smoke_test_source_df",
    target="product",
    spark_session=spark,
)

display(warehouse_df)

# - Warehouse query uses the Fabric Warehouse Spark connector with SQL pushdown
# - Prefer this for Warehouse data because filters/projections are pushed down before Spark receives the data.
warehouse_query_df = read_warehouse_query(
    "SELECT TOP 3 * FROM dbo.smoke_test_source_df",
    target="product",
    spark_session=spark,
)

display(warehouse_query_df)


# FabricOps v0.2.0 onwards

## Preview- Example standardized data profiling of your defined source

This profile is local exploratory output only. It does not update metadata tables, create catalogue evidence, approve governance, or enforce guardrails.

It lists each column in your DataFrame, its data type, total row count, null count, distinct count, and simple min/max evidence where applicable.


In [ ]:
display(
        profile_dataframe(
        source_df,               # the df you want to profile
        table_name="orders_csv", # Give your df a meaningful and readable name
        )
)

## Browse and inspect the data catalogue

Select any dataset catalogued in the current environment, then load its Spark DataFrames in native Fabric result cells.


In [ ]:
data_catalogue_view = widget_view_data_catalogue(
    target="metadata",
    spark_session=spark,
)


In [ ]:
views = data_catalogue_view["get_views"]()
catalogue_df = views["catalogue"]
profile_df = views["profile"]
frequency_df = views["frequency"]


In [ ]:
display(catalogue_df)


In [ ]:
display(profile_df)
display(frequency_df)
